# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 15: Autenticación Stateless con JSON Web Tokens (JWT)
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Codificación Base64Url y firma HMAC-SHA256*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Generación y verificación de tokens JWT con PyJWT* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas con tokens alterados o firma inválida*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion15.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/noble/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.189.186), connection timed out Could not connect to ppa.launchpadcontent.net:443 (185.125.189.187), connection timed out Could not connect to ppa.launchpadcontent.net:443 (185.125.189.188), connection timed out
W: Some index files failed to download. They have been ignored, or old ones used instead.
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Codificación Base64Url y firma HMAC-SHA256

In [2]:
%%writefile app_artesano.py
import base64
import hmac
import hashlib
import json
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano")

class PayloadRequest(BaseModel):
    dato: str

class LoginRequest(BaseModel):
    username: str
    password: str

@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

# Nuestra clave secreta
SECRET_KEY = b"mi_clave_super_secreta_dab_2201"

def base64url_encode(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode('utf-8').rstrip("=")

@app.post("/api/login-manual")
def login_manual(user: LoginRequest):
    if user.username == "admin" and user.password == "123456":
        # crea el header
        header = {"alg": "HS256", "typ": "JWT"}
        header_json = json.dumps(header, separators=(',', ':')).encode('utf-8')
        header_b64 = base64url_encode(header_json)

        #  el payload(datos delk usuario)
        payload = {"sub": user.username, "role": "admin"}
        payload_json = json.dumps(payload, separators=(',', ':')).encode('utf-8')
        payload_b64 = base64url_encode(payload_json)

        #  Con esto se crea la firma
        mensaje = f"{header_b64}.{payload_b64}".encode('utf-8')
        firma = hmac.new(SECRET_KEY, mensaje, hashlib.sha256).digest()
        firma_b64 = base64url_encode(firma)

        # une
        token = f"{header_b64}.{payload_b64}.{firma_b64}"

        return {
            "status": "success",
            "access_token": token,
            "token_type": "bearer",
            "mensaje": "Token generado manualmente con HMAC-SHA256"
        }
    raise HTTPException(status_code=401, detail="Credenciales inválidas")


Writing app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [ ]:
%%writefile app_ia.py
from fastapi import FastAPI

# Pega aquí el código generado por la IA para auditarlo


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas con tokens alterados o firma inválida

In [3]:
import threading
import time
import uvicorn
import httpx
from app_artesano import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 15 ===\n")
r = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta Servidor: HTTP {r.status_code} - {r.json()}")
assert r.status_code == 200
print("PASO ✅ - Test Harness HTTP ejecutado con exito.")


=== AUDITORIA CRITICA WEB: SESION 15 ===

Respuesta Servidor: HTTP 200 - {'status': 'success', 'mensaje': 'Procesado: Prueba Test Harness'}
PASO ✅ - Test Harness HTTP ejecutado con exito.
